In [1]:
import time
notebook_start_time = time.time()

# 1. Importing libraries

In [2]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 98.8 MB/s eta 0:00:00


In [3]:
!uv pip install datasets transformers huggingface_hub torch tqdm optuna

Using Python 3.12.13 environment at: /usr
Resolved 80 packages in 538ms
Prepared 2 packages in 34ms
Installed 2 packages in 9ms
 + colorlog==6.10.1
 + optuna==4.8.0


In [4]:
!uv pip install mamba-ssm --no-build-isolation-package mamba-ssm

Using Python 3.12.13 environment at: /usr
Resolved 56 packages in 114ms
Prepared 1 package in 8ms
Installed 1 package in 2ms
Prepared 1 package without build isolation in 25.71s
Installed 1 package in 2ms
 + mamba-ssm==2.3.1
 + ninja==1.13.0


In [5]:
import itertools
import random
import string

from mamba_ssm import Mamba

import torch
from datasets import load_dataset

from tqdm import tqdm

import math
import torch.nn as nn
import torch.nn.functional as F

import optuna

In [6]:
NUM_EPOCH = 40

# 2. AI models

In [7]:
def parallel_associative_scan(gates, values):
    L = gates.shape[0]
    a = gates.clone()
    b = values.clone()
    stride = 1
    while stride < L:
        idx = torch.arange(stride, L, device=gates.device)
        prev = idx - stride
        new_b = a[idx] * b[prev] + b[idx]
        new_a = a[idx] * a[prev]
        a[idx] = new_a
        b[idx] = new_b
        stride *= 2
    return b

In [8]:
class MambaBlock(nn.Module):
    def __init__(self, d_model, N, dt_min=0.001, dt_max=0.1, expand=2):
        super().__init__()
        self.d_model = d_model
        self.N = N
        self.dt_min = dt_min
        self.dt_max = dt_max
        self.d_inner = d_model * expand

        A_log_init = torch.tensor([math.log(n + 1) for n in range(N)], dtype=torch.float32)
        self.A_log = nn.Parameter(A_log_init)
        self.b_c = nn.Parameter(torch.zeros(N))

        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.dt_rank = max(1, d_model // 16)
        self.delta_proj    = nn.Linear(self.d_inner, self.dt_rank, bias=False)
        self.delta_proj_up = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        nn.init.constant_(self.delta_proj_up.bias, math.log(math.expm1(1)))

        self.B_proj = nn.Linear(self.d_inner, N, bias=False)
        self.C_proj = nn.Linear(self.d_inner, N, bias=False)

        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=4,
                                groups=self.d_inner, padding=3, bias=True)

    def forward(self, x_seq):
        seq_len = x_seq.shape[0]
        xz = self.in_proj(x_seq)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_branch = x_branch.T.unsqueeze(0)
        x_branch = F.silu(self.conv1d(x_branch)[..., :seq_len])
        x_branch = x_branch.squeeze(0).T

        delta = F.softplus(self.delta_proj_up(self.delta_proj(x_branch))).clamp(self.dt_min, self.dt_max)
        B_all = self.B_proj(x_branch)
        C_all = self.C_proj(x_branch)
        A_diag = -torch.exp(self.A_log)

        A_bar = torch.exp(A_diag.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1))
        B_bar = ((A_bar - 1) / A_diag.unsqueeze(0).unsqueeze(0)) * B_all.unsqueeze(1)
        values = B_bar * x_branch.unsqueeze(-1)

        h_all = parallel_associative_scan(A_bar, values)
        h_all = h_all + self.b_c
        y = (h_all * C_all.unsqueeze(1)).sum(dim=-1)

        output = y * F.silu(z_branch)
        return self.out_proj(output)

In [9]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

In [10]:
class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, **kwargs):
        super().__init__()
        self.norm  = RMSNorm(d_model)
        self.mamba = MambaBlock(d_model, N, **kwargs)

    def forward(self, x):
        return x + self.mamba(self.norm(x))

In [11]:
# ---- Mamba encoder backbone (shared by all wrappers) ----

class MambaEncoder(nn.Module):
    """Embedding -> [ResidualBlock x n_layers] -> RMSNorm"""
    def __init__(self, vocab_size, d_model=128, n_layers=2, N=16, expand=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, expand=expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)

    def forward(self, input_ids):
        """input_ids: (B, L) -> (B, L, d_model)"""
        x = self.embedding(input_ids)
        outputs = []
        for b in range(x.shape[0]):
            h = x[b]
            for layer in self.layers:
                h = layer(h)
            h = self.norm_f(h)
            outputs.append(h)
        return torch.stack(outputs, dim=0)


# ---- Wrapper 1: Token-level prediction ----
# For MQAR, Induction Heads, MAD recall tasks, Memorization

class MambaTokenPredictor(nn.Module):
    """Per-position logits over vocab. Use with ignore_index=-100 in CE loss."""
    def __init__(self, vocab_size, d_model=128, n_layers=2, N=16):
        super().__init__()
        self.encoder = MambaEncoder(vocab_size, d_model, n_layers, N)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.encoder(input_ids)          # (B, L, d_model)
        return self.head(h)                  # (B, L, vocab_size)


# ---- Wrapper 2: Sequence-to-fixed-output ----
# For Selective Copying, Compression (input L tokens -> output K tokens)

class MambaSeqToFixed(nn.Module):
    """Reads full sequence, outputs K token predictions from the last K hidden states."""
    def __init__(self, vocab_size, output_len, d_model=128, n_layers=2, N=16):
        super().__init__()
        self.output_len = output_len
        self.encoder = MambaEncoder(vocab_size, d_model, n_layers, N)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.encoder(input_ids)                    # (B, L, d_model)
        h_tail = h[:, -self.output_len:, :]            # (B, K, d_model)
        return self.head(h_tail)                       # (B, K, vocab_size)


# ---- Wrapper 3: Sequence classification ----
# For Parity

class MambaClassifier(nn.Module):
    """Mean-pool -> classify."""
    def __init__(self, vocab_size, num_classes, d_model=128, n_layers=2, N=16):
        super().__init__()
        self.encoder = MambaEncoder(vocab_size, d_model, n_layers, N)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids):
        h = self.encoder(input_ids)          # (B, L, d_model)
        pooled = h.mean(dim=1)               # (B, d_model)
        return self.classifier(pooled)       # (B, num_classes)

In [12]:
def train_loop(model, train_inputs, train_labels, epochs=NUM_EPOCH, lr=1e-3, batch_size=32, ignore_index=-100):
    """Simple training loop with progress bars. Returns final loss."""
    device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    n = train_inputs.shape[0]

    for epoch in tqdm(range(1, epochs + 1), desc="Epochs", unit="epoch"):
        model.train()
        perm = torch.randperm(n)
        total_loss = 0
        steps = 0

        batch_iter = tqdm(
            range(0, n, batch_size),
            desc=f"  Epoch {epoch:3d}",
            leave=False,
            unit="batch",
        )
        for i in batch_iter:
            idx = perm[i:i+batch_size]
            x = train_inputs[idx].to(device)
            y = train_labels[idx].to(device)

            logits = model(x)
            logits_flat = logits.reshape(-1, logits.size(-1))
            labels_flat = y.reshape(-1)
            loss = F.cross_entropy(logits_flat, labels_flat, ignore_index=ignore_index)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
            batch_iter.set_postfix(loss=f"{loss.item():.4f}")

        if epoch % 5 == 0 or epoch == 1:
            tqdm.write(f"  Epoch {epoch:3d}  loss = {total_loss/steps:.4f}")

    return total_loss / steps

In [13]:
def eval_accuracy(model, inputs, labels, batch_size=64, ignore_index=-100):
    """Compute accuracy over non-ignored positions."""
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for i in tqdm(range(0, inputs.shape[0], batch_size), desc="Evaluating", leave=False, unit="batch"):
            x = inputs[i:i+batch_size].to(device)
            y = labels[i:i+batch_size].to(device)
            logits = model(x)
            preds = logits.argmax(dim=-1)

            mask = (y != ignore_index)
            correct += (preds[mask] == y[mask]).sum().item()
            total += mask.sum().item()

    acc = correct / total if total > 0 else 0.0
    return acc

# 3. Benchmarks to ensure design works - Create training, validation and test datasets for each benchmark

## 3.0 Configurations

In [14]:
COMBINATIONS = {
    "combination_1": {"d_model": 64,  "n_layers": 4, "N": 8, "lr": 1e-3},
    "combination_2": {"d_model": 128, "n_layers": 2, "N": 8, "lr": 5e-4},
}

## 3.1 Selective Copying

 - Problem: Standard copying tasks could be solved with fixed convolution kernels using positional tricks. HackerNoon Randomizing token positions forces content-aware selection.

 - Why beyond LRA: Tests input-dependent gating — the core innovation in Mamba/S6 — which LRA doesn't isolate.

In [15]:
def generate_selective_copying(
    batch_size=64,
    seq_len=128,
    num_tokens_to_copy=8,
    vocab_size=16,
    blank_token=0,
    marker_token=None,
    seed=42
):
    """
    Selective Copying: a sequence has `num_tokens_to_copy` meaningful tokens
    scattered at random positions among blank tokens. The model must output
    only the meaningful tokens in order.

    Unlike fixed-interval copying, positions are random — so the model must
    use content-aware (selective) reasoning, not just fixed offsets.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if marker_token is None:
        marker_token = vocab_size  # special token outside vocab

    inputs = torch.full((batch_size, seq_len), blank_token, dtype=torch.long)
    labels = torch.zeros(batch_size, num_tokens_to_copy, dtype=torch.long)

    for b in range(batch_size):
        # Pick random positions for the meaningful tokens
        positions = sorted(random.sample(range(seq_len - 1), num_tokens_to_copy))
        tokens = torch.randint(1, vocab_size, (num_tokens_to_copy,))

        for i, pos in enumerate(positions):
            inputs[b, pos] = tokens[i]
            labels[b, i] = tokens[i]

        # Append a marker token at the end to signal "now reproduce"
        inputs[b, -1] = marker_token

    return inputs, labels

inputs, labels = generate_selective_copying(batch_size=32, seq_len=64, num_tokens_to_copy=6)
print("=== Selective Copying ===")
print(f"Inputs shape:  {inputs.shape}")
print(f"Labels shape:  {labels.shape}")
print(f"Sample input:  {inputs[0]}")
print(f"Expected copy: {labels[0]}")

=== Selective Copying ===
Inputs shape:  torch.Size([32, 64])
Labels shape:  torch.Size([32, 6])
Sample input:  tensor([ 0, 13,  0,  0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0,  0,  2,  0,  5,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  7,  0,  0,  0,  0,  0,  0,  6,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0, 16])
Expected copy: tensor([13,  3,  2,  5,  7,  6])


In [16]:
NUM_COPY = 6

train_inp, train_lab = generate_selective_copying(batch_size=512, seq_len=128, num_tokens_to_copy=NUM_COPY, seed=42)
val_inp,   val_lab   = generate_selective_copying(batch_size=128, seq_len=128, num_tokens_to_copy=NUM_COPY, seed=99)
test_inp,  test_lab  = generate_selective_copying(batch_size=128, seq_len=128, num_tokens_to_copy=NUM_COPY, seed=7)

In [17]:
results_selcopy = {}

for combo_name, cfg in COMBINATIONS.items():
    print(f"\n{'='*50}\nSelective Copying — {combo_name}: {cfg}\n{'='*50}")

    model = MambaSeqToFixed(vocab_size=17, output_len=NUM_COPY,
                            d_model=cfg["d_model"], n_layers=cfg["n_layers"], N=cfg["N"])
    train_loop(model, train_inp, train_lab, epochs=NUM_EPOCH, lr=cfg["lr"], batch_size=32, ignore_index=-1)
    acc = eval_accuracy(model, test_inp, test_lab, ignore_index=-1)
    print(f"Selective Copying test accuracy: {acc:.4f}")

    results_selcopy[combo_name] = {"config": cfg, "test_acc": acc}


Selective Copying — combination_1: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.001}


Epochs:   2%|▎         | 1/40 [00:52<34:20, 52.84s/epoch]

  Epoch   1  loss = 2.8704



Epochs:  12%|█▎        | 5/40 [04:31<31:53, 54.68s/epoch]

  Epoch   5  loss = 2.7097


...


Epochs:  88%|████████▊ | 35/40 [25:17<03:35, 43.12s/epoch]

  Epoch  35  loss = 1.6259



Epochs: 100%|██████████| 40/40 [28:53<00:00, 43.33s/epoch]


  Epoch  40  loss = 1.4782


Selective Copying test accuracy: 0.3255


## 3.2 Induction Heads

 - Problem: Needed to understand the mechanism behind in-context learning. arXiv Found that induction heads ([A][B]...[A]→[B]) are likely the primary circuit driving ICL.

 - Why beyond LRA: Probes the specific two-layer copy-and-recall circuit that underpins in-context learning — critical for evaluating SSMs vs. transformers.

In [18]:
def generate_induction_heads(
    batch_size=64,
    seq_len=128,
    vocab_size=64,
    num_triggers=4,
    seed=42
):
    """
    Induction Head task: earlier in the sequence, pattern "A B" appears.
    Later, "A" appears again — the model must predict "B".

    This tests the two-layer attention circuit:
      Layer 1 attends to the previous token of the current query.
      Layer 2 copies the token that followed the previous occurrence.
    """
    random.seed(seed)
    torch.manual_seed(seed)

    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = torch.full((batch_size, seq_len), -100, dtype=torch.long)  # -100 = ignore

    for b in range(batch_size):
        # Create trigger pairs (A, B) in the first half
        pair_positions = sorted(random.sample(range(0, seq_len // 2 - 1), num_triggers))
        pairs = []
        for pos in pair_positions:
            a = random.randint(1, vocab_size - 1)
            btoken = random.randint(1, vocab_size - 1)
            inputs[b, pos] = a
            inputs[b, pos + 1] = btoken
            pairs.append((a, btoken))

        # Place query "A" tokens in the second half; label is "B"
        query_positions = sorted(random.sample(
            range(seq_len // 2, seq_len - 1), num_triggers
        ))
        for i, qpos in enumerate(query_positions):
            a, btoken = pairs[i]
            inputs[b, qpos] = a
            labels[b, qpos] = btoken  # model should predict B here

    return inputs, labels

inputs, labels = generate_induction_heads(batch_size=32, seq_len=64, num_triggers=3)
print("=== Induction Heads ===")
print(f"Inputs shape:  {inputs.shape}")
print(f"Labels shape:  {labels.shape}")

# Show the trigger positions (where label != -100)
sample_idx = 0
trigger_mask = labels[sample_idx] != -100
trigger_pos = trigger_mask.nonzero().squeeze()
print(f"Query positions: {trigger_pos.tolist()}")
print(f"Expected tokens: {labels[sample_idx, trigger_mask].tolist()}")

=== Induction Heads ===
Inputs shape:  torch.Size([32, 64])
Labels shape:  torch.Size([32, 64])
Query positions: [35, 53, 55]
Expected tokens: [18, 15, 48]


In [19]:
train_inp, train_lab = generate_induction_heads(batch_size=512, seq_len=128, num_triggers=4, seed=42)
val_inp,   val_lab   = generate_induction_heads(batch_size=128, seq_len=128, num_triggers=4, seed=99)
test_inp,  test_lab  = generate_induction_heads(batch_size=128, seq_len=128, num_triggers=4, seed=7)

In [20]:
results_induction = {}

for combo_name, cfg in COMBINATIONS.items():
    print(f"\n{'='*50}\nInduction Heads — {combo_name}: {cfg}\n{'='*50}")

    model = MambaTokenPredictor(vocab_size=64,
                                d_model=cfg["d_model"], n_layers=cfg["n_layers"], N=cfg["N"])
    train_loop(model, train_inp, train_lab, epochs=NUM_EPOCH, lr=cfg["lr"], batch_size=32)
    acc = eval_accuracy(model, test_inp, test_lab)
    print(f"Induction Heads test accuracy: {acc:.4f}")

    results_induction[combo_name] = {"config": cfg, "test_acc": acc}


Induction Heads — combination_1: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.001}


Epochs:   2%|▎         | 1/40 [00:49<31:53, 49.06s/epoch]

  Epoch   1  loss = 4.3075



Epochs:  12%|█▎        | 5/40 [04:11<29:27, 50.51s/epoch]

  Epoch   5  loss = 3.7953


...


Epochs:  88%|████████▊ | 35/40 [22:10<03:09, 37.81s/epoch]

  Epoch  35  loss = 0.0244



Epochs: 100%|██████████| 40/40 [25:20<00:00, 38.02s/epoch]


  Epoch  40  loss = 0.0179


Induction Heads test accuracy: 0.0098


# 4. Complete results

In [25]:
print("\n" + "=" * 70)
print("  RESULTS SUMMARY — BOTH COMBINATIONS")
print("=" * 70)

for name, results in [
    ("Selective Copying", results_selcopy),
    ("Induction Heads",   results_induction),
]:
    print(f"\n  {name}")
    for combo_name, r in results.items():
        print(f"    {combo_name}: test_acc={r['test_acc']:.4f}  config={r['config']}")

print("\n" + "=" * 70)


  RESULTS SUMMARY — BOTH COMBINATIONS

  Selective Copying
    combination_1: test_acc=0.3411  config={'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.001}
    combination_2: test_acc=0.3255  config={'d_model': 128, 'n_layers': 2, 'N': 8, 'lr': 0.0005}

  Induction Heads
    combination_1: test_acc=0.0234  config={'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.001}
    combination_2: test_acc=0.0098  config={'d_model': 128, 'n_layers': 2, 'N': 8, 'lr': 0.0005}



In [26]:
notebook_end_time = time.time()
total_time = notebook_end_time - notebook_start_time

# Convert to minutes and seconds for easier reading
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"Total 'Run All' execution time: {minutes} minutes and {seconds} seconds")

Total 'Run All' execution time: 142 minutes and 11 seconds
